In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
# Write your code here
#custom dats set
#from torch.utils.data import Dataset
import os
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch.utils.data import DataLoader


train_transform = transforms.Compose([ #aug for train phase only
    transforms.Resize((32, 32)), #resize to 32x32
    transforms.RandomRotation(15),
    transforms.ToTensor()
])

transform = transforms.Compose([ #general everyerybody can use it
    transforms.Resize((32, 32)), #resize to 32x32
    transforms.ToTensor()
])
train_dir = os.path.join(path, "/kaggle/input/q1-stage-3-2026/PlantVillage/train")
test_dir = os.path.join(path, "/kaggle/input/q1-stage-3-2026/PlantVillage/test")


from torchvision.datasets import ImageFolder
#dataset
# Automatically handles everything if folders are named correctly
train_dataset = ImageFolder(root=train_dir,  transform=train_transform) #train data set
test_dataset  = ImageFolder(root=test_dir,  transform=transform) #test dataset

#dataloader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)





In [ ]:
from PIL import Image

# dispaly one img image
image_path = os.path.join(path,"/kaggle/input/q1-stage-3-2026/PlantVillage/train/Potato___Early_blight/001187a0-57ab-4329-baff-e7246a9edeb0___RS_Early.B 8178.JPG")
image = Image.open(image_path)

# Display the image
image

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import torch

#display sample imgs
# Create a function to visualize samples
def visualize_samples(dataset, num_samples=8, title="Dataset Samples"):
    """
    Visualize random samples from a dataset.

    Args:
        dataset: PyTorch Dataset object
        num_samples: Number of samples to display
        title: Title for the plot
    """
    # Select random indices
    indices = random.sample(range(len(dataset)), num_samples)

    # Calculate grid size
    cols = 4
    rows = (num_samples + cols - 1) // cols

    # Create the plot
    fig, axes = plt.subplots(rows, cols, figsize=(12, 3 * rows))
    axes = axes.flatten()

    for i, idx in enumerate(indices):
        # Get image and label
        image, label = dataset[idx]
        # Convert tensor to numpy for display
        if isinstance(image, torch.Tensor):
            image = image.permute(1, 2, 0).numpy()  # CHW -> HWC
        # Get class name
        class_name = dataset.classes[label]
        # Display image
        axes[i].imshow(image)
        axes[i].set_title(f"{class_name}\n(Label: {label})", fontsize=10)
        axes[i].axis('off')
    # Hide any unused subplots
    for i in range(num_samples, len(axes)):
        axes[i].axis('off')

    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
visualize_samples(train_dataset, num_samples=8, title="Training Dataset Samples")

In [ ]:
visualize_samples(test_dataset, num_samples=8, title="Testing Dataset Samples")

In [ ]:
images, labels = next(iter(train_loader))
print(f"Batch shape: {images.shape}, Labels: {labels}")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # it is better to run CNNs with GPUs for faster computation
device

In [ ]:
# Write your code here
import torch.nn as nn
class BotatoCNN(nn.Module):
    def __init__(self, num_classes=3,dropout_rate=0.2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),  # [B,32,28,28]
            nn.ReLU(),
            nn.MaxPool2d(2),                             # [B,32,14,14]
            nn.Conv2d(32,64, kernel_size=3, padding=1),  # [B,64,14,14]
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),                            # [B,64,7,7]
            nn.BatchNorm1d(num_classes),
            nn.Tanh(),

        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            # Calculate input features (channels × height × width)
            # After 2 MaxPool2d(2): 28 → 14 → 7
            nn.Linear(28 * 14 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
         #Pass x through features
        x = self.features(x)
        # result through classifier
        x = self.classifier(x)
        return x


In [ ]:
# Write your code here
from tqdm import tqdm

def accuracy_from_logits(logits, labels):
    #  Get predicted class indices

    preds = torch.argmax(logits, dim=1)
    #  Calculate and return accuracy
    return (preds == labels).float().mean().item()

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, total_acc = 0.0, 0.0

    for batch in tqdm(loader):
        images, labels = batch['image'], batch['label']
        images, labels = images.to(device), labels.to(device)

        # TO-DO: Zero the gradients
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_acc += accuracy_from_logits(logits.detach(), labels)

    return total_loss / len(loader), total_acc / len(loader)


In [ ]:
# Write your code here
import torch.optim as optim


# Initialize the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BotatoCNN(num_classes=3).to(device)

# Print model summary
print(model)

# Define loss function and optimizer
criterion = nn.MSELoss()  # Measure reconstruction quality
optimizer = optim.AdamW(model.parameters(), lr=1e-4)  # AdamW optimizer
num_epochs = 20 # Number of epochs

# Store losses for plotting
train_losses = []

# Training loop
for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer)
    train_losses.append(train_loss)

    print(f"Epoch {epoch+1}/{num_epochs}, Loss = {train_loss:.4f}")

In [ ]:
# Write your code here
